# Phase 1 on Colab (T4 GPU)

Runs preprocessing + SigLIP-2 feature extraction + the zero-shot baseline on Colab's T4.

## Before running, upload to Google Drive `MyDrive/mvp/`:
1. **The latest scripts** (copy from your Mac's `mvp/` folder):
   `attributes.py`, `extract_features.py`, `evaluate_zeroshot.py`
2. **The dataset as one zip**: on your Mac, zip the `data/PA-100K` folder into `PA100K.zip`
   and upload it to `MyDrive/mvp/PA100K.zip`.

Then in VS Code: open this notebook -> Select Kernel -> **Colab** -> pick a **T4 GPU** runtime -> run cells top to bottom.

In [ ]:
# 1. Confirm we have a GPU (should say Tesla T4)
!nvidia-smi -L

In [ ]:
# 2. Install extra deps (torch is already on Colab with CUDA)
!pip install -q transformers sentencepiece scipy

In [ ]:
# 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. Copy the scripts to fast local disk and go there
import shutil, os
PROJ = '/content/drive/MyDrive/mvp'
os.makedirs('/content/work', exist_ok=True)
for f in ['attributes.py', 'extract_features.py', 'evaluate_zeroshot.py']:
    shutil.copy(f'{PROJ}/{f}', f'/content/work/{f}')
%cd /content/work
print('scripts:', os.listdir('.'))

In [ ]:
# 5. Unzip PA-100K to fast local disk (NOT read directly from Drive)
!cp '/content/drive/MyDrive/mvp/PA100K.zip' /content/PA100K.zip
!unzip -q /content/PA100K.zip -d /content/data
# show the layout so we can confirm the paths
!find /content/data -maxdepth 3 -type d

In [ ]:
# 6. Set the exact dataset paths (matches your folder layout)
ANN = '/content/data/PA-100K/annotation/annotation.mat'
IMG = '/content/data/PA-100K/data/release_data/release_data'
import os
assert os.path.exists(ANN), 'annotation.mat not found -- check the unzipped path above'
assert os.path.isdir(IMG), 'image folder not found -- check the unzipped path above'
print('OK:', ANN, '|', IMG)

In [ ]:
# 7. Extract features -- TEST split first (quick check)
!python extract_features.py --ann "$ANN" --img_dir "$IMG" --splits test --out /content/features

In [ ]:
# 8. Zero-shot baseline on the test split  ->  baseline numbers
!python evaluate_zeroshot.py --features /content/features --split test --groups

In [ ]:
# 9. Full extraction: train + val (T4 makes this fast)
!python extract_features.py --ann "$ANN" --img_dir "$IMG" --splits train val --out /content/features

In [ ]:
# 10. Save the feature cache back to Drive (Colab wipes /content on disconnect!)
!mkdir -p '/content/drive/MyDrive/mvp/features'
!cp /content/features/* '/content/drive/MyDrive/mvp/features/'
print('features saved to Drive -> MyDrive/mvp/features')
!ls -la '/content/drive/MyDrive/mvp/features'